In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

# Memuat dataset
df = pd.read_csv(
    "../dataset/laptop_price.csv",
    encoding="latin1"
)

print("Dataset berhasil dimuat!")
print("Ukuran dataset:", df.shape)

Dataset berhasil dimuat!
Ukuran dataset: (1303, 13)


In [2]:
# Membuat salinan agar dataset asli tidak berubah
df_clean = df.copy()

# Menghapus kolom identitas
df_clean = df_clean.drop(columns=["laptop_ID"])

df_clean.head()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00
3,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,2537.45
4,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60


In [3]:
df_clean["Ram_GB"] = (
    df_clean["Ram"]
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

df_clean[["Ram", "Ram_GB"]].head()

,Ram,Ram_GB
0,8GB,8.0
1,8GB,8.0
2,8GB,8.0
3,16GB,16.0
4,8GB,8.0


In [4]:
df_clean["Weight_kg"] = (
    df_clean["Weight"]
    .str.extract(r"([\d.]+)", expand=False)
    .astype(float)
)

df_clean[["Weight", "Weight_kg"]].head()

,Weight,Weight_kg
0,1.37kg,1.37
1,1.34kg,1.34
2,1.86kg,1.86
3,1.83kg,1.83
4,1.37kg,1.37


In [5]:
print("RAM yang gagal dikonversi:", df_clean["Ram_GB"].isna().sum())
print("Berat yang gagal dikonversi:", df_clean["Weight_kg"].isna().sum())

df_clean[["Ram", "Ram_GB", "Weight", "Weight_kg"]].describe()

RAM yang gagal dikonversi: 0
Berat yang gagal dikonversi: 0


,Ram_GB,Weight_kg
count,1303.000000,1303.000000
mean,8.382195,2.038734
std,5.084665,0.665475
min,2.000000,0.690000
25%,4.000000,1.500000
50%,8.000000,2.040000
75%,8.000000,2.300000
max,64.000000,4.700000


In [6]:
# Memeriksa seluruh variasi kapasitas penyimpanan
print("Jumlah variasi Memory:", df_clean["Memory"].nunique())

display(
    df_clean["Memory"]
    .value_counts()
    .to_frame(name="Jumlah")
)

Jumlah variasi Memory: 39


,Jumlah
Memory,
256GB SSD,412
1TB HDD,223
500GB HDD,132
512GB SSD,118
128GB SSD + 1TB HDD,94
128GB SSD,76
256GB SSD + 1TB HDD,73
32GB Flash Storage,38
2TB HDD,16


In [7]:
def extract_memory(memory):
    memory = str(memory).upper()

    result = {
        "SSD_GB": 0,
        "HDD_GB": 0,
        "Flash_GB": 0,
        "Hybrid_GB": 0
    }

    pattern = r"(\d+(?:\.\d+)?)\s*(TB|GB)\s*(SSD|HDD|FLASH STORAGE|HYBRID)"

    matches = re.findall(pattern, memory)

    for capacity, unit, storage_type in matches:
        capacity = float(capacity)

        if unit == "TB":
            capacity *= 1024

        if storage_type == "SSD":
            result["SSD_GB"] += capacity
        elif storage_type == "HDD":
            result["HDD_GB"] += capacity
        elif storage_type == "FLASH STORAGE":
            result["Flash_GB"] += capacity
        elif storage_type == "HYBRID":
            result["Hybrid_GB"] += capacity

    return pd.Series(result)


memory_features = df_clean["Memory"].apply(extract_memory)

df_clean = pd.concat(
    [df_clean, memory_features],
    axis=1
)

df_clean[["Memory", "SSD_GB", "HDD_GB", "Flash_GB", "Hybrid_GB"]].head(15)

,Memory,SSD_GB,HDD_GB,Flash_GB,Hybrid_GB
0,128GB SSD,128.0,0.0,0.0,0.0
1,128GB Flash Storage,0.0,0.0,128.0,0.0
2,256GB SSD,256.0,0.0,0.0,0.0
3,512GB SSD,512.0,0.0,0.0,0.0
4,256GB SSD,256.0,0.0,0.0,0.0
5,500GB HDD,0.0,500.0,0.0,0.0
6,256GB Flash Storage,0.0,0.0,256.0,0.0
7,256GB Flash Storage,0.0,0.0,256.0,0.0
8,512GB SSD,512.0,0.0,0.0,0.0
9,256GB SSD,256.0,0.0,0.0,0.0


In [8]:
storage_columns = [
    "SSD_GB",
    "HDD_GB",
    "Flash_GB",
    "Hybrid_GB"
]

print("Jumlah nilai kosong:")
print(df_clean[storage_columns].isna().sum())

print("\nStatistik kapasitas penyimpanan:")
display(df_clean[storage_columns].describe())

Jumlah nilai kosong:
SSD_GB       0
HDD_GB       0
Flash_GB     0
Hybrid_GB    0
dtype: int64

Statistik kapasitas penyimpanan:


,SSD_GB,HDD_GB,Flash_GB,Hybrid_GB
count,1303.000000,1303.000000,1303.000000,1303.000000
mean,184.027629,422.477360,4.555641,9.034536
std,188.268689,528.006298,30.274090,94.738890
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,256.000000,0.000000,0.000000,0.000000
75%,256.000000,1024.000000,0.000000,0.000000
max,1024.000000,2048.000000,512.000000,1024.000000


In [9]:
# Memeriksa apakah ada data Memory yang tidak berhasil diekstrak

storage_columns = [
    "SSD_GB",
    "HDD_GB",
    "Flash_GB",
    "Hybrid_GB"
]

# Menghitung total kapasitas yang berhasil diekstrak
df_clean["Total_Storage_GB"] = df_clean[storage_columns].sum(axis=1)

# Mencari baris dengan seluruh hasil ekstraksi bernilai nol
unmatched = df_clean[df_clean["Total_Storage_GB"] == 0]

print("Jumlah data yang tidak berhasil diekstrak:", len(unmatched))

if len(unmatched) > 0:
    print("\nFormat Memory yang belum berhasil dikenali:")
    display(unmatched["Memory"].value_counts())
else:
    print("Seluruh data Memory berhasil diekstrak!")

# Memeriksa distribusi jenis penyimpanan
print("\nJumlah laptop berdasarkan jenis penyimpanan:")

for col in storage_columns:
    print(f"{col}: {(df_clean[col] > 0).sum()} laptop")

Jumlah data yang tidak berhasil diekstrak: 0
Seluruh data Memory berhasil diekstrak!

Jumlah laptop berdasarkan jenis penyimpanan:
SSD_GB: 843 laptop
HDD_GB: 576 laptop
Flash_GB: 75 laptop
Hybrid_GB: 12 laptop


In [10]:
# Memeriksa variasi prosesor
print("Jumlah variasi CPU:", df_clean["Cpu"].nunique())

display(
    df_clean["Cpu"]
    .value_counts()
    .to_frame(name="Jumlah")
)

Jumlah variasi CPU: 118


,Jumlah
Cpu,
Intel Core i5 7200U 2.5GHz,190
Intel Core i7 7700HQ 2.8GHz,146
Intel Core i7 7500U 2.7GHz,134
Intel Core i7 8550U 1.8GHz,73
Intel Core i5 8250U 1.6GHz,72
...,...
Intel Core i5 7200U 2.70GHz,1
Intel Core M M7-6Y75 1.2GHz,1
Intel Core M 6Y54 1.1GHz,1


In [11]:
# Mengidentifikasi produsen prosesor
def identify_cpu_brand(cpu):
    cpu = str(cpu).lower()

    if "intel" in cpu:
        return "Intel"
    elif "amd" in cpu:
        return "AMD"
    else:
        return "Other"


df_clean["CPU_Brand"] = df_clean["Cpu"].apply(identify_cpu_brand)

print("Distribusi produsen CPU:")
display(df_clean["CPU_Brand"].value_counts())

print("\nContoh nama CPU:")
display(df_clean[["Cpu", "CPU_Brand"]].head(20))

Distribusi produsen CPU:


CPU_Brand
Intel    1240
AMD        62
Other       1
Name: count, dtype: int64


Contoh nama CPU:


,Cpu,CPU_Brand
0,Intel Core i5 2.3GHz,Intel
1,Intel Core i5 1.8GHz,Intel
2,Intel Core i5 7200U 2.5GHz,Intel
3,Intel Core i7 2.7GHz,Intel
4,Intel Core i5 3.1GHz,Intel
5,AMD A9-Series 9420 3GHz,AMD
6,Intel Core i7 2.2GHz,Intel
7,Intel Core i5 1.8GHz,Intel
8,Intel Core i7 8550U 1.8GHz,Intel
9,Intel Core i5 8250U 1.6GHz,Intel


In [12]:
# Mengelompokkan nama CPU berdasarkan keluarga prosesor

def identify_cpu_family(cpu):
    cpu = str(cpu).lower()

    if "core i9" in cpu:
        return "Intel Core i9"
    elif "core i7" in cpu:
        return "Intel Core i7"
    elif "core i5" in cpu:
        return "Intel Core i5"
    elif "core i3" in cpu:
        return "Intel Core i3"
    elif "core m" in cpu:
        return "Intel Core M"
    elif "celeron" in cpu:
        return "Intel Celeron"
    elif "pentium" in cpu:
        return "Intel Pentium"
    elif "atom" in cpu:
        return "Intel Atom"
    elif "ryzen 9" in cpu:
        return "AMD Ryzen 9"
    elif "ryzen 7" in cpu:
        return "AMD Ryzen 7"
    elif "ryzen 5" in cpu:
        return "AMD Ryzen 5"
    elif "ryzen 3" in cpu:
        return "AMD Ryzen 3"
    elif "amd" in cpu:
        return "AMD Other"
    else:
        return "Other"


df_clean["CPU_Family"] = df_clean["Cpu"].apply(identify_cpu_family)

print("Distribusi keluarga CPU:")
display(
    df_clean["CPU_Family"]
    .value_counts()
    .to_frame(name="Jumlah Laptop")
)

Distribusi keluarga CPU:


,Jumlah Laptop
CPU_Family,
Intel Core i7,527
Intel Core i5,423
Intel Core i3,136
Intel Celeron,88
AMD Other,62
Intel Pentium,30
Intel Core M,19
Intel Atom,13
Other,5


In [13]:
import re

# 1. Skor keluarga CPU berdasarkan pemetaan heuristik
cpu_family_score = {
    "Intel Core i3": 3,
    "Intel Core i5": 5,
    "Intel Core i7": 7,
    "Intel Core i9": 9,
    "Intel Core M": 2,
    "Intel Celeron": 1,
    "Intel Pentium": 2,
    "Intel Atom": 1,
    "AMD Other": 3,
    "Other": 1
}

df_clean["CPU_Family_Score"] = (
    df_clean["CPU_Family"].map(cpu_family_score)
)

# 2. Mengambil generasi Intel Core dari nama CPU
def extract_cpu_generation(cpu):
    cpu = str(cpu)

    match = re.search(
        r"Core i[3579]\s+(\d{4,5})",
        cpu,
        re.IGNORECASE
    )

    if match:
        model = match.group(1)

        # Untuk nomor model 5 digit, dua digit pertama
        # menunjukkan generasi pada sebagian besar model Core
        if len(model) == 5:
            return int(model[:2])

        return int(model[0])

    return np.nan


df_clean["CPU_Generation"] = (
    df_clean["Cpu"].apply(extract_cpu_generation)
)

# 3. Mengambil kecepatan clock dalam GHz
def extract_cpu_clock(cpu):
    match = re.search(
        r"(\d+(?:\.\d+)?)\s*GHz",
        str(cpu),
        re.IGNORECASE
    )

    return float(match.group(1)) if match else np.nan


df_clean["CPU_Base_GHz"] = (
    df_clean["Cpu"].apply(extract_cpu_clock)
)

# Menampilkan hasil
display(
    df_clean[
        [
            "Cpu",
            "CPU_Family",
            "CPU_Family_Score",
            "CPU_Generation",
            "CPU_Base_GHz"
        ]
    ].head(20)
)

,Cpu,CPU_Family,CPU_Family_Score,CPU_Generation,CPU_Base_GHz
0,Intel Core i5 2.3GHz,Intel Core i5,5,NaN,2.3
1,Intel Core i5 1.8GHz,Intel Core i5,5,NaN,1.8
2,Intel Core i5 7200U 2.5GHz,Intel Core i5,5,7.0,2.5
3,Intel Core i7 2.7GHz,Intel Core i7,7,NaN,2.7
4,Intel Core i5 3.1GHz,Intel Core i5,5,NaN,3.1
5,AMD A9-Series 9420 3GHz,AMD Other,3,NaN,3.0
6,Intel Core i7 2.2GHz,Intel Core i7,7,NaN,2.2
7,Intel Core i5 1.8GHz,Intel Core i5,5,NaN,1.8
8,Intel Core i7 8550U 1.8GHz,Intel Core i7,7,8.0,1.8
9,Intel Core i5 8250U 1.6GHz,Intel Core i5,5,8.0,1.6


In [14]:
# Memeriksa kelengkapan fitur CPU

cpu_features = [
    "CPU_Family_Score",
    "CPU_Generation",
    "CPU_Base_GHz"
]

print("Jumlah nilai kosong pada fitur CPU:")
display(df_clean[cpu_features].isna().sum())

print("\nPersentase kelengkapan fitur CPU:")
display(
    (df_clean[cpu_features].notna().mean() * 100)
    .round(2)
    .to_frame(name="Kelengkapan (%)")
)

Jumlah nilai kosong pada fitur CPU:


CPU_Family_Score      0
CPU_Generation      243
CPU_Base_GHz          0
dtype: int64


Persentase kelengkapan fitur CPU:


,Kelengkapan (%)
CPU_Family_Score,100.00
CPU_Generation,81.35
CPU_Base_GHz,100.00


In [15]:
print("Distribusi generasi CPU:")
display(
    df_clean["CPU_Generation"]
    .value_counts(dropna=False)
    .sort_index()
)

Distribusi generasi CPU:


CPU_Generation
6.0    310
7.0    603
8.0    147
NaN    243
Name: count, dtype: int64

In [16]:
# Menentukan fitur numerik awal
feature_columns = [
    "Price_euros",
    "Ram_GB",
    "Weight_kg",
    "SSD_GB",
    "HDD_GB",
    "CPU_Family_Score",
    "CPU_Base_GHz"
]

# Membuat dataset fitur
X = df_clean[feature_columns].copy()

# Memeriksa struktur dan kelengkapan
print("Ukuran dataset fitur:", X.shape)

print("\nTipe data:")
print(X.dtypes)

print("\nNilai kosong:")
print(X.isna().sum())

print("\nStatistik deskriptif:")
display(X.describe().T)

Ukuran dataset fitur: (1303, 7)

Tipe data:
Price_euros         float64
Ram_GB              float64
Weight_kg           float64
SSD_GB              float64
HDD_GB              float64
CPU_Family_Score      int64
CPU_Base_GHz        float64
dtype: object

Nilai kosong:
Price_euros         0
Ram_GB              0
Weight_kg           0
SSD_GB              0
HDD_GB              0
CPU_Family_Score    0
CPU_Base_GHz        0
dtype: int64

Statistik deskriptif:


,count,mean,std,min,25%,50%,75%,max
Price_euros,1303.0,1123.686992,699.009043,174.00,599.0,977.00,1487.88,6099.0
Ram_GB,1303.0,8.382195,5.084665,2.00,4.0,8.00,8.00,64.0
Weight_kg,1303.0,2.038734,0.665475,0.69,1.5,2.04,2.30,4.7
SSD_GB,1303.0,184.027629,188.268689,0.00,0.0,256.00,256.00,1024.0
HDD_GB,1303.0,422.477360,528.006298,0.00,0.0,0.00,1024.00,2048.0
CPU_Family_Score,1303.0,5.066769,1.965757,1.00,3.0,5.00,7.00,7.0
CPU_Base_GHz,1303.0,2.298772,0.506340,0.90,2.0,2.50,2.70,3.6


In [17]:
# Membuat salinan fitur sebelum transformasi
X_transformed = X.copy()

# Fitur dengan distribusi yang perlu ditransformasi
log_columns = [
    "Price_euros",
    "Ram_GB",
    "SSD_GB",
    "HDD_GB"
]

# Transformasi log1p
for col in log_columns:
    X_transformed[col] = np.log1p(X_transformed[col])

print("Transformasi logaritmik berhasil!")
display(X_transformed.head())

Transformasi logaritmik berhasil!


,Price_euros,Ram_GB,Weight_kg,SSD_GB,HDD_GB,CPU_Family_Score,CPU_Base_GHz
0,7.200940,2.197225,1.37,4.859812,0.0,5,2.3
1,6.802328,2.197225,1.34,0.000000,0.0,5,1.8
2,6.356108,2.197225,1.86,5.549076,0.0,5,2.5
3,7.839309,2.833213,1.83,6.240276,0.0,7,2.7
4,7.498094,2.197225,1.37,5.549076,0.0,5,3.1


In [18]:
from sklearn.preprocessing import MinMaxScaler

# Inisialisasi scaler
scaler = MinMaxScaler()

# Normalisasi seluruh fitur
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_transformed),
    columns=feature_columns,
    index=X_transformed.index
)

print("Normalisasi berhasil!")
print("Ukuran data:", X_scaled.shape)

display(X_scaled.head())

Normalisasi berhasil!
Ukuran data: (1303, 7)


,Price_euros,Ram_GB,Weight_kg,SSD_GB,HDD_GB,CPU_Family_Score,CPU_Base_GHz
0,0.573361,0.357182,0.169576,0.701024,0.0,0.666667,0.518519
1,0.461116,0.357182,0.162095,0.000000,0.0,0.666667,0.333333
2,0.335465,0.357182,0.291771,0.800450,0.0,0.666667,0.592593
3,0.753120,0.563956,0.284289,0.900155,0.0,1.000000,0.666667
4,0.657037,0.357182,0.169576,0.800450,0.0,0.666667,0.814815


In [19]:
print("Nilai minimum setiap fitur:")
display(X_scaled.min())

print("\nNilai maksimum setiap fitur:")
display(X_scaled.max())

print("\nStatistik setelah normalisasi:")
display(X_scaled.describe().T)

Nilai minimum setiap fitur:


Price_euros         0.0
Ram_GB              0.0
Weight_kg           0.0
SSD_GB              0.0
HDD_GB              0.0
CPU_Family_Score    0.0
CPU_Base_GHz        0.0
dtype: float64


Nilai maksimum setiap fitur:


Price_euros         1.0
Ram_GB              1.0
Weight_kg           1.0
SSD_GB              1.0
HDD_GB              1.0
CPU_Family_Score    1.0
CPU_Base_GHz        1.0
dtype: float64


Statistik setelah normalisasi:


,count,mean,std,min,25%,50%,75%,max
Price_euros,1303.0,0.471858,0.175435,0.0,0.346960,0.484539,0.602883,1.0
Ram_GB,1303.0,0.334383,0.148687,0.0,0.166080,0.357182,0.357182,1.0
Weight_kg,1303.0,0.336343,0.165954,0.0,0.201995,0.336658,0.401496,1.0
SSD_GB,1303.0,0.515036,0.385643,0.0,0.000000,0.800450,0.800450,1.0
HDD_GB,1303.0,0.393923,0.444053,0.0,0.000000,0.000000,0.909161,1.0
CPU_Family_Score,1303.0,0.677795,0.327626,0.0,0.333333,0.666667,1.000000,1.0
CPU_Base_GHz,1303.0,0.518064,0.187534,0.0,0.407407,0.592593,0.666667,1.0


In [20]:
# Menyimpan hasil preprocessing
X_scaled.to_csv(
    "../dataset/laptop_features_scaled.csv",
    index=False
)

print("Data hasil preprocessing berhasil disimpan!")
print("Ukuran data:", X_scaled.shape)

Data hasil preprocessing berhasil disimpan!
Ukuran data: (1303, 7)
